Combine all preprocessed metric files into two canonical tidy CSVs:
- `data/tidy/drawings.csv` — one row per participant × sea-animal category, drawing metrics
- `data/tidy/utterances.csv` — one row per participant × sea-animal category, utterance metrics

Both files include adults (AD001–AD020) and children (BD025+) with `is_adult`/`is_child` flags for analysis-level filtering.
Run top-to-bottom to overwrite the tidy files.

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

In [2]:
SEA_ANIMALS = ["crab", "octopus", "seahorse", "shark", "turtle", "whale"]

# Source files
FEATURES_CSV  = "../data/clean/half_and_full_utterance_features.csv"
AGES_CSV      = "../data/clean/participant_age_table.csv"
HIGHRES_CSV   = "../data/clean/drawing_highres_clean.csv"
HALF_EMB_CSV  = "../data/embeddings/half_time_sketch_embeddings/half_time_clip_embeddings.csv"
DRAW_ADULTS   = "../data/preprocessed/draw_cossim_adults.csv"
DRAW_TOSELF   = "../data/preprocessed/draw_cossim_toself.csv"
TALK_ADULTS   = "../data/preprocessed/talk_cossim_adults.csv"
TALK_TOSELF   = "../data/preprocessed/talk_cossim_toself.csv"

# Outputs
DRAWINGS_OUT   = "../data/tidy/drawings.csv"
UTTERANCES_OUT = "../data/tidy/utterances.csv"

## Shared spine

`half_and_full_utterance_features.csv` is the source of truth for participant × category membership, interference flags, exists flags, and hand-coded feature lists. All downstream metric files are left-joined onto it.

In [3]:
features = pd.read_csv(FEATURES_CSV)
features = features.rename(columns={"SubID": "participant_id"})
features["participant_id"] = features["participant_id"].str.strip()
features["target_category"] = features["target_category"].str.strip().str.lower()

# Non-pilot participants only: AD001–AD020 and BD025–BD125
def is_valid(pid):
    pid = str(pid).strip()
    if pid.startswith("AD"):
        try: return 1 <= int(pid[2:]) <= 20
        except: return False
    if pid.startswith("BD"):
        try: return int(pid[2:]) >= 25
        except: return False
    return False

features = features[features["participant_id"].apply(is_valid)].copy()
features = features[features["target_category"].isin(SEA_ANIMALS)].copy()

# Merge age, derive flags; drop participants with missing age (data collection error)
ages = pd.read_csv(AGES_CSV)
features = features.merge(ages, on="participant_id", how="left")
n_before = len(features)
features = features.dropna(subset=["age_months"]).copy()
print(f"Dropped {n_before - len(features)} rows with missing age_months")

# Boolean flags so R reads them as logical (not double)
features["is_adult"] = features["participant_id"].str.startswith("AD")
features["is_child"] = features["participant_id"].str.startswith("BD")

print("Spine:", features.shape)
print(features["is_adult"].value_counts().rename({False: "child", True: "adult"}))

Dropped 9 rows with missing age_months
Spine: (638, 21)
is_adult
child    518
adult    120
Name: count, dtype: int64


## Drawings tidy

In [4]:
draw = features[[
    "participant_id", "target_category", "age_months", "age_yrs",
    "is_adult", "is_child", "draw_exists", "draw_interference",
    "draw_full_features", "draw_half_features"
]].copy()

# Drawing-level metadata (trial_duration, num_strokes, etc.)
highres = pd.read_csv(HIGHRES_CSV)
highres["target_category"] = (
    highres["text1"]
    .str.replace(r"^(a|an|this)_", "", regex=True)
    .str.replace("_", " ")
    .str.lower().str.strip()
)
highres = highres[highres["target_category"].isin(SEA_ANIMALS)]
draw = draw.merge(
    highres[["participant_id", "target_category",
              "trial_duration", "num_strokes", "draw_duration", "mean_intensity", "bounding_box"]],
    on=["participant_id", "target_category"], how="left"
)

# Cosine similarity to adult mean (children only; adults will be NaN)
draw = draw.merge(
    pd.read_csv(DRAW_ADULTS)[["participant_id", "target_category", "draw_cossim_adults", "draw_half_cossim_adults"]],
    on=["participant_id", "target_category"], how="left"
)

# Half-time stroke metadata
half_emb = pd.read_csv(HALF_EMB_CSV)
half_emb["target_category"] = half_emb["category"].str.replace(r"^(a|an)\s+", "", regex=True).str.strip()
half_emb = half_emb[half_emb["target_category"].isin(SEA_ANIMALS)].copy()
half_emb["_bd_num"] = pd.to_numeric(
    half_emb["participant_id"].str.extract(r"^BD(\d+)$")[0], errors="coerce"
)
half_emb = half_emb[
    half_emb["participant_id"].str.startswith("AD") | (half_emb["_bd_num"] >= 25)
].drop(columns="_bd_num")
draw = draw.merge(
    half_emb[["participant_id", "target_category", "n_strokes_found", "half_k"]],
    on=["participant_id", "target_category"], how="left"
)

# Within-subject cosine similarity (draw_cossim_{cat}, draw_probability_{cat})
toself_cols = (
    [f"draw_cossim_{c}" for c in SEA_ANIMALS] +
    [f"draw_probability_{c}" for c in SEA_ANIMALS]
)
draw = draw.merge(
    pd.read_csv(DRAW_TOSELF)[["participant_id", "target_category"] + toself_cols],
    on=["participant_id", "target_category"], how="left"
)

# Final column order
draw = draw[[
    "participant_id", "target_category", "age_months", "age_yrs",
    "is_adult", "is_child", "draw_exists", "draw_interference",
    "trial_duration", "num_strokes", "draw_duration", "mean_intensity", "bounding_box",
    "draw_cossim_adults", "draw_half_cossim_adults",
    "n_strokes_found", "half_k",
] + toself_cols + ["draw_full_features", "draw_half_features"]]

draw = draw.sort_values(["participant_id", "target_category"]).reset_index(drop=True)
print("drawings shape:", draw.shape)
draw.head()

drawings shape: (638, 31)


,participant_id,target_category,age_months,age_yrs,is_adult,is_child,draw_exists,draw_interference,trial_duration,num_strokes,...,draw_cossim_turtle,draw_cossim_whale,draw_probability_crab,draw_probability_octopus,draw_probability_seahorse,draw_probability_shark,draw_probability_turtle,draw_probability_whale,draw_full_features,draw_half_features
0,AD001,crab,229.0,19.13,True,False,1,0,17.843,10.0,...,0.174802,-0.053971,NaN,0.180589,0.188921,0.205612,0.236634,0.188244,"antannae, pinchers, legs, shell","pinchers, legs, shell"
1,AD001,octopus,229.0,19.13,True,False,1,0,15.795,2.0,...,-0.030418,-0.009841,0.186853,NaN,0.233353,0.176817,0.199416,0.203562,"eight tentacles, bulbous head",bulbous head
2,AD001,seahorse,229.0,19.13,True,False,1,0,10.183,4.0,...,-0.072934,-0.066269,0.195283,0.233125,NaN,0.188460,0.190928,0.192205,"head, long body, long nose, curly tail","head, long nose"
3,AD001,shark,229.0,19.13,True,False,1,0,14.193,6.0,...,0.217222,0.389400,0.187092,0.155497,0.165899,NaN,0.224651,0.266860,"long body, pointy nose, eyes, side fins, dorsa...","long body, pointy nose, tail"
4,AD001,turtle,229.0,19.13,True,False,1,0,16.793,10.0,...,NaN,0.218565,0.213534,0.173916,0.166677,0.222787,NaN,0.223086,"head, eyes, shell, legs, tail","head, shell"


In [5]:
# NaN audit before saving
nulls = draw.isnull().sum()
print(nulls[nulls > 0])

trial_duration                 9
num_strokes                    9
draw_duration                  9
mean_intensity                 9
bounding_box                   9
draw_cossim_adults           129
draw_half_cossim_adults      128
n_strokes_found                8
half_k                         8
draw_cossim_crab             131
draw_cossim_octopus          130
draw_cossim_seahorse         134
draw_cossim_shark            138
draw_cossim_turtle           134
draw_cossim_whale            135
draw_probability_crab        131
draw_probability_octopus     130
draw_probability_seahorse    134
draw_probability_shark       138
draw_probability_turtle      134
draw_probability_whale       135
draw_full_features            89
draw_half_features           125
dtype: int64


In [6]:
Path(DRAWINGS_OUT).parent.mkdir(parents=True, exist_ok=True)
draw.to_csv(DRAWINGS_OUT, index=False)
print(f"Saved {draw.shape[0]} rows → {DRAWINGS_OUT}")

Saved 638 rows → ../data/tidy/drawings.csv


## Utterances tidy

In [7]:
utt = features[[
    "participant_id", "target_category", "age_months", "age_yrs",
    "is_adult", "is_child", "utterance_exists", "utterance_interference",
    "utterance_ns", "utterance_masked",
    "utterance_full_masked_filtered", "utterance_half_masked_filtered",
    "utterance_full_visual_features", "utterance_full_semantic_features",
    "utterance_half_visual_features", "utterance_half_semantic_features"
]].copy()

# Cosine similarity to adult mean (children only; adults will be NaN)
utt = utt.merge(
    pd.read_csv(TALK_ADULTS)[["participant_id", "target_category",
                               "utterance_full_cossim_adults", "utterance_half_cossim_adults"]],
    on=["participant_id", "target_category"], how="left"
)

# Within-subject cosine similarity (utterance_cossim_{cat}, utterance_probability_{cat})
talk_toself_cols = (
    [f"utterance_cossim_{c}" for c in SEA_ANIMALS] +
    [f"utterance_probability_{c}" for c in SEA_ANIMALS]
)
utt = utt.merge(
    pd.read_csv(TALK_TOSELF)[["participant_id", "target_category"] + talk_toself_cols],
    on=["participant_id", "target_category"], how="left"
)

# Final column order
utt = utt[[
    "participant_id", "target_category", "age_months", "age_yrs",
    "is_adult", "is_child", "utterance_exists", "utterance_interference",
    "utterance_ns", "utterance_masked",
    "utterance_full_masked_filtered", "utterance_half_masked_filtered",
    "utterance_full_cossim_adults", "utterance_half_cossim_adults",
] + talk_toself_cols + [
    "utterance_full_visual_features", "utterance_full_semantic_features",
    "utterance_half_visual_features", "utterance_half_semantic_features"
]]

utt = utt.sort_values(["participant_id", "target_category"]).reset_index(drop=True)
print("utterances shape:", utt.shape)
utt.head()

utterances shape: (638, 30)


,participant_id,target_category,age_months,age_yrs,is_adult,is_child,utterance_exists,utterance_interference,utterance_ns,utterance_masked,...,utterance_probability_crab,utterance_probability_octopus,utterance_probability_seahorse,utterance_probability_shark,utterance_probability_turtle,utterance_probability_whale,utterance_full_visual_features,utterance_full_semantic_features,utterance_half_visual_features,utterance_half_semantic_features
0,AD001,crab,229.0,19.13,True,False,1,0,"It has two claws and two legs, and it is usual...","It has two claws and two legs, and it is usual...",...,NaN,0.241211,0.228940,0.162421,0.161487,0.205940,"pinchers, legs, color, eyes, small",crawl,"pinchers, legs, color, small",NaN
1,AD001,octopus,229.0,19.13,True,False,1,0,It has eight legs and it's usually also bright...,It has eight legs and it's usually also bright...,...,0.236183,NaN,0.234496,0.152811,0.169667,0.206843,"eight tentacles, color","camoflauge, sticky","eight tentacles, color",NaN
2,AD001,seahorse,229.0,19.13,True,False,1,0,Seahorses looks small and it's usually like a ...,It looks small and it's usually like a bright ...,...,0.204764,0.214198,NaN,0.186580,0.143804,0.250655,"color, long nose, curly tail, small",NaN,"small, color",NaN
3,AD001,shark,229.0,19.13,True,False,1,0,It's a big sea creature and it's usually grey....,It's a big sea creature and it's usually grey....,...,0.170499,0.163825,0.218983,NaN,0.188340,0.258352,"big, color, teeth, fins, tail",sea animal,"big, color",sea animal
4,AD001,turtle,229.0,19.13,True,False,1,0,It has a hard shell and four legs which helps ...,It has a hard shell and four legs which helps ...,...,0.192246,0.206284,0.191408,0.213592,NaN,0.196470,"shell, legs, pattern","swim, hard","shell, legs","swim, hard"


In [8]:
# NaN audit before saving
nulls = utt.isnull().sum()
print(nulls[nulls > 0])

utterance_ns                         77
utterance_masked                     91
utterance_full_masked_filtered       91
utterance_half_masked_filtered       91
utterance_full_cossim_adults        207
utterance_half_cossim_adults        207
utterance_cossim_crab               218
utterance_cossim_octopus            209
utterance_cossim_seahorse           209
utterance_cossim_shark              218
utterance_cossim_turtle             200
utterance_cossim_whale              196
utterance_probability_crab          218
utterance_probability_octopus       209
utterance_probability_seahorse      209
utterance_probability_shark         218
utterance_probability_turtle        200
utterance_probability_whale         196
utterance_full_visual_features      121
utterance_full_semantic_features    217
utterance_half_visual_features      178
utterance_half_semantic_features    358
dtype: int64


In [9]:
utt.to_csv(UTTERANCES_OUT, index=False)
print(f"Saved {utt.shape[0]} rows → {UTTERANCES_OUT}")

Saved 638 rows → ../data/tidy/utterances.csv
